# CoughSense — Dataset Inspection

This notebook documents the initial inspection of the dataset used for CoughSense.

The purpose of this stage is to understand the structure of the audio and metadata before building the preprocessing and machine learning pipeline.

## 1. Import Required Libraries

We import `pandas` for working with tabular data and `Path` for handling dataset file paths in a platform-independent way.

In [ ]:

import pandas as pd
from pathlib import Path

print("Python Environment is Working!")

## 2. Locate the Dataset

The Coswara-derived dataset is kept outside the GitHub repository because it contains a large amount of audio data.

We store its location in `DATASET_DIR` so that the rest of the notebook can access the dataset without hard-coding the path repeatedly.

In [ ]:
from pathlib import Path

DATASET_DIR = Path(r"C:\Users\siddh\my-dataset")

print("Dataset path:", DATASET_DIR)
print("Dataset exists:", DATASET_DIR.exists())

## 3. Inspect Dataset Structure

We recursively list the files inside the dataset directory.

This allows us to verify how the downloaded dataset is organized before reading any data. In particular, we want to identify the training, validation, and test audio files and their corresponding metadata files.

In [ ]:
for path in DATASET_DIR.rglob("*"):
    if path.is_file():
        print(path.relative_to(r"C:\Users\siddh\my-dataset"))

## 4. Inspect an Audio Parquet File

We inspect one training Parquet shard to understand how individual audio recordings are stored.

We are checking:
- the number of records in the shard
- the available columns
- how each recording is identified
- which fields contain the audio and associated information

In [ ]:
audio_path = DATASET_DIR / "audio" / "train" / "audio-train-00001.parquet"

audio_df = pd.read_parquet(audio_path)

print("Shape:", audio_df.shape)
print("\nColumns:")
print(audio_df.columns.tolist())

## 5. Inspect the Audio Representation

The `audio` column is inspected to determine how the recordings are encoded inside the Parquet file.

Understanding this representation is necessary before we write the audio preprocessing pipeline that will eventually convert the recordings into Mel spectrograms.

In [ ]:
print("Audio type:", type(audio_df.loc[0, "audio"]))
print("Audio value preview:", audio_df.loc[0, "audio"])
print("Audio_type:", audio_df.loc[0, "audio_type"])
print("Participant ID:", audio_df.loc[0, "participant_id"])

## 6. Available Audio Recording Types

Each participant can provide several types of audio recordings.

We inspect the `audio_type` distribution to determine which recordings are available and which types are most relevant to the CoughSense respiratory screening pipeline.

In [ ]:
print(audio_df["audio_type"].value_counts(dropna=False))

## 7. Participant-Level Recording Structure

Each participant can have multiple audio recordings.

We check the number of unique participants and the number of recordings associated with each participant.

This is important because participant identity must be preserved when working with the dataset and especially when evaluating the model, so that recordings from the same participant do not cause data leakage between dataset splits.

In [ ]:
print("Unique participants:", audio_df["participant_id"].nunique())
print("Total recordings:", len(audio_df))

print("\nRecordings per participant:")
print(audio_df["participant_id"].value_counts().head(10))

## 8. Inspect the Metadata

The metadata contains participant-level information such as health status, symptoms, pre-existing conditions, and recording information.

We inspect the metadata structure before deciding which fields can be used as prediction targets, contextual features, or filtering criteria.

In [ ]:
metadata_path = DATASET_DIR / "metadata" / "train.parquet"

metadata_df = pd.read_parquet(metadata_path)

print("Shape:", metadata_df.shape)
print("\nColumns:")
print(metadata_df.columns.tolist())

## 9. Match Audio Records to Metadata

Audio recordings identify participants using `participant_id`, while the metadata identifies participants using `id`.

We compare these identifiers to verify that audio recordings can be correctly associated with the corresponding participant-level medical annotations.

In [ ]:
audio_ids = set(audio_df["participant_id"].astype(str))
metadata_ids = set(metadata_df["id"].astype(str))

matching_ids = audio_ids & metadata_ids

print("Audio participants:", len(audio_ids))
print("Metadata participants:", len(metadata_ids))
print("Matching participants:", len(matching_ids))

## 10. Initial Target Availability

We inspect several potential CoughSense prediction targets to determine how many participants have an explicit positive annotation for each target.

Missing values are not automatically treated as negative labels. They represent unavailable or unrecorded information and must be handled carefully during dataset construction.

In [ ]:
target_columns = [
    "asthma",
    "pneumonia",
    "cough",
    "fever",
    "loss_of_smell",
]

for col in target_columns:
    print(f"\n--- {col} ---")
    print(metadata_df[col].value_counts(dropna=False))

## 11. Examine Target Encoding

We inspect the unique values of the candidate target columns.

This determines how the original metadata represents positive, negative, and missing information before we create clean CoughSense labels.

In [ ]:
for col in ["asthma", "pneumonia", "cough", "fever", "loss_of_smell"]:
    print(f"\n--- {col} ---")
    print("Unique values:", metadata_df[col].unique())

## 12. Inspect Additional Metadata

Additional fields are examined to understand their possible roles in the CoughSense system.

Some variables may become prediction targets, some may be valid contextual inputs for the MLP, while others may need to be excluded because they are identifiers, potential sources of data leakage, or not available when a new user is screened.

In [ ]:
for col in [

    "covid_status",

    "test_status",

    "smoker",

    "others_resp",

    "others_preexist",

    "vacc",

]:

    print(f"\n--- {col} ---")

    print(metadata_df[col].value_counts(dropna=False))

## 13. Inspect Metadata Encoding

Some metadata fields contain inconsistent representations such as `y`/`n`, `True`/`False`, or missing values.

We inspect these encodings before normalization so that the derived CoughSense dataset can represent the information consistently without incorrectly treating missing values as negative responses.

In [ ]:
for col in ["smoker", "others_resp", "others_preexist", "vacc"]:
    print(f"\n--- {col} ---")
    print(metadata_df[col].value_counts(dropna=False).to_string())

## 14. Check Participant Overlap Between Splits

The dataset provides separate training, validation, and test metadata.

We verify that no participant appears in more than one split.

This is critical for preventing data leakage because each participant has multiple audio recordings. A participant appearing in both training and testing could make the model appear more accurate than it really is.

In [ ]:
train_metadata = pd.read_parquet(
    DATASET_DIR / "metadata" / "train.parquet"
)

val_metadata = pd.read_parquet(
    DATASET_DIR / "metadata" / "val.parquet"
)

test_metadata = pd.read_parquet(
    DATASET_DIR / "metadata" / "test.parquet"
)

train_ids = set(train_metadata["id"].astype(str))
val_ids = set(val_metadata["id"].astype(str))
test_ids = set(test_metadata["id"].astype(str))

print("Train participants:", len(train_ids))
print("Validation participants:", len(val_ids))
print("Test participants:", len(test_ids))

print("\nTrain ∩ Validation:", len(train_ids & val_ids))
print("Train ∩ Test:", len(train_ids & test_ids))
print("Validation ∩ Test:", len(val_ids & test_ids))

## 15. Dataset Inspection Summary

The initial inspection established that:

- Audio recordings are stored as WAV data inside Parquet files.
- Each recording has a `participant_id`.
- Each participant has multiple audio recording types.
- The metadata uses participant IDs that can be matched to the audio records.
- The metadata contains several potential respiratory and symptom-related targets.
- Missing target annotations must not automatically be treated as negative labels.
- The dataset provides participant-disjoint training, validation, and test splits.

These findings will be used to design the CoughSense-specific dataset and multi-target training pipeline.

In [ ]:
candidate_targets = [
    "asthma",
    "pneumonia",
    "fever",
    "loss_of_smell",
    "covid_status",
]

for col in candidate_targets:
    print(f"\n--- {col} ---")
    print(metadata_df[col].value_counts(dropna=False).to_string())

## 16. Candidate Target Distribution

We examine the distribution of the candidate CoughSense targets across the complete training metadata.

The goal is to determine how much labelled data is available for each prediction task before creating the derived multi-target dataset.

Missing values are treated as unlabeled observations rather than negative labels.

In [ ]:
candidate_targets = [
    "asthma",
    "pneumonia",
    "fever",
    "loss_of_smell",
    "covid_status",
]

for col in candidate_targets:
    print(f"\n--- {col} ---")
    print(metadata_df[col].value_counts(dropna=False).to_string())

## 17. COVID Status Distribution

COVID status is represented as a multi-class categorical variable rather than a simple binary label.

We inspect the complete distribution before deciding whether CoughSense should use the original categories directly or derive a simpler screening-oriented target.

In [ ]:
print(metadata_df["covid_status"].value_counts(dropna=False).to_string())

## 18. Audio Quality Analysis

The metadata contains quality information for each recording type.

We inspect the quality fields for cough and breathing recordings to understand how recording quality is represented and determine whether low-quality recordings should be filtered during dataset preparation.

In [ ]:
quality_columns = [
    "cough_heavy_quality",
    "cough_shallow_quality",
    "breathing_deep_quality",
    "breathing_shallow_quality",
]

for col in quality_columns:
    print(f"\n--- {col} ---")
    print(metadata_df[col].value_counts(dropna=False).to_string())

## 19. Audio Quality Filtering Decision

The dataset documentation defines the audio quality labels as:

- `0` — Bad: poor quality with significant noise or distortion; excluded from classification.
- `1` — Good: acceptable quality for analysis.
- `2` — Excellent: high-quality, clear audio.

For CoughSense, recordings with quality `0` will be excluded from model training and evaluation. Recordings with quality `1` or `2` will be retained.

The quality score is used as a data-quality filter rather than as a model input feature.

## 20. Cough Recording Selection

CoughSense will use the `cough-heavy` and `cough-shallow` recordings as the primary audio inputs for the cough-based screening model.

Recordings with quality `0` are excluded according to the dataset documentation. Recordings with quality `1` or `2` are retained.

The participant-level train, validation, and test splits will be preserved so that recordings from the same participant do not cross dataset boundaries.

In [ ]:
cough_types = ["cough-heavy", "cough-shallow"]

cough_df = audio_df[audio_df["audio_type"].isin(cough_types)].copy()

print("Total cough recordings:", len(cough_df))

for audio_type in cough_types:
    subset = cough_df[cough_df["audio_type"] == audio_type].copy()

    quality_col = (
        "cough_heavy_quality"
        if audio_type == "cough-heavy"
        else "cough_shallow_quality"
    )

    # Get the quality score from the participant-level metadata.
    subset = subset.merge(
        metadata_df[["id", quality_col]],
        left_on="participant_id",
        right_on="id",
        how="left"
    )

    print(f"\n--- {audio_type} ---")
    print("Total:", len(subset))
    print(
        subset[quality_col]
        .value_counts(dropna=False)
        .sort_index()
        .to_string()
    )

## 21. Compare Audio and Metadata Quality Scores

The audio Parquet contains a general `quality_score`, while the participant metadata contains recording-specific quality fields such as `cough_heavy_quality` and `cough_shallow_quality`.

We compare these values for the same recordings to determine whether they represent the same quality assessment and which field should be used for CoughSense quality filtering.

In [ ]:
# Merge the cough recordings with their participant-level metadata.
comparison_df = cough_df.merge(
    metadata_df[
        [
            "id",
            "cough_heavy_quality",
            "cough_shallow_quality",
        ]
    ],
    left_on="participant_id",
    right_on="id",
    how="left",
)

# Select the metadata quality column corresponding to each recording type.
comparison_df["metadata_quality"] = comparison_df.apply(
    lambda row: (
        row["cough_heavy_quality"]
        if row["audio_type"] == "cough-heavy"
        else row["cough_shallow_quality"]
    ),
    axis=1,
)

print(
    comparison_df[
        [
            "participant_id",
            "audio_type",
            "quality_score",
            "metadata_quality",
        ]
    ].head(20).to_string(index=False)
)

print("\nQuality score combinations:")
print(
    comparison_df[
        ["quality_score", "metadata_quality"]
    ]
    .value_counts(dropna=False)
    .sort_index()
    .to_string()
)

## 21. Quality Score Consistency

The audio Parquet contains a `quality_score` for each recording, while the metadata contains recording-specific quality fields.

For the inspected cough recordings, the two quality measurements agree exactly:

- `0` → Bad
- `1` → Good
- `2` → Excellent

Therefore, CoughSense will use the audio-level `quality_score` when filtering recordings. Recordings with a score of `0` will be excluded, while scores of `1` and `2` will be retained.

## 22. Cough Recording Representation

Each participant has two cough recording types: `cough-heavy` and `cough-shallow`.

For the initial CoughSense model, each acceptable-quality cough recording will be treated as an individual audio training sample while retaining the participant-level medical labels.

The existing participant-level train, validation, and test splits will be preserved. We will not randomly redistribute individual recordings across these splits.

The participant remains the independent subject for evaluation, even though multiple cough recordings may contribute samples.

## 23. CoughSense Target and Input Design

CoughSense is designed as a multi-task respiratory pre-screening system.

Potential prediction targets include:

- Asthma
- Pneumonia
- Fever
- Loss of smell
- COVID-related status

Potential contextual inputs for the MLP include information that could be collected from a new user, such as:

- Age
- Gender
- Smoking status

Ground-truth medical outcomes must not be provided to the model as input features when the same outcomes are prediction targets, in order to prevent data leakage.

The `cough` metadata field is not treated as a prediction target because the system already receives a cough recording as its primary input.

The final target set will be confirmed after evaluating label availability and class balance across the complete dataset.

## 24. Inspect MLP Input Features

The CoughSense multimodal model may use selected user-provided contextual information alongside the cough audio.

We inspect age, gender, and smoking status to understand their distributions, missing values, and categorical encodings before deciding how they should be represented for the MLP.

Only information that can realistically be collected from a new user will be considered as an inference-time input.

## 24.1 Inspect Metadata Field Definitions

Some demographic fields in the metadata use abbreviated column names.

Before using these fields as CoughSense MLP inputs, we verify their definitions from the dataset documentation rather than assuming their meanings from the column names.

In [ ]:
print(metadata_df[["a", "g"]].head(10).to_string())

## 24.2 Verify Demographic Field Definitions

The metadata uses abbreviated column names for some demographic variables.

The observed values suggest that:
- `a` represents age
- `g` represents gender

These definitions will be verified using the dataset documentation before the fields are incorporated into the CoughSense MLP.

## 24.3 Inspect Demographic and Smoking Inputs

The candidate contextual inputs for the CoughSense MLP are age (`a`), gender (`g`), and smoking status (`smoker`).

We inspect their distributions and missing values before defining the encoding used by the model.

These features are intended to represent information that could be collected directly from a new user during screening.

In [ ]:
mlp_input_columns = ["a", "g", "smoker"]

for col in mlp_input_columns:
    print(f"\n--- {col} ---")
    print("Data type:", metadata_df[col].dtype)
    print("Missing:", metadata_df[col].isna().sum())
    print(metadata_df[col].value_counts(dropna=False).to_string())

### 24.4 Gender Distribution

We inspect the complete gender distribution and missing-value count to determine how the categorical gender feature should be encoded for the MLP.

In [ ]:
print("Gender data type:", metadata_df["g"].dtype)
print("Gender missing:", metadata_df["g"].isna().sum())
print("\nGender distribution:")
print(metadata_df["g"].value_counts(dropna=False).to_string())

### 24.4 Gender Distribution

The metadata contains three gender categories: `male`, `female`, and `other`.

There are no missing gender values in the training metadata. All three categories will be retained when preparing the CoughSense MLP input rather than excluding the low-frequency `other` category.

Gender will later be encoded as a categorical feature for the MLP.

### 24.5 Smoking Status Distribution

The smoking-status field contains multiple representations of the same underlying categories, including `y`/`n` and `True`/`False`, as well as missing values.

We inspect the complete distribution before defining a consistent representation for the CoughSense MLP.

In [ ]:
print("Smoker data type:", metadata_df["smoker"].dtype)
print("Smoker missing:", metadata_df["smoker"].isna().sum())
print("\nSmoker distribution:")
print(metadata_df["smoker"].value_counts(dropna=False).to_string())

### 24.5 Smoking Status Handling

Smoking status is represented using multiple values:

- `y` and `True` indicate a smoker.
- `n` and `False` indicate a non-smoker.
- Missing values indicate that smoking status is unknown.

For the CoughSense MLP, the explicitly recorded values will be normalized into a consistent representation. Missing smoking status will remain distinguishable from a known non-smoker rather than being treated as negative.

## 25. Create the CoughSense Dataset

We create a task-specific dataset by combining cough recordings with the corresponding participant-level metadata.

Only `cough-heavy` and `cough-shallow` recordings are used. Recordings with quality `0` are excluded, while quality `1` and `2` are retained.

The existing participant-level train, validation, and test splits are preserved.

Medical target fields remain missing when the original dataset does not contain an annotation. Missing targets will be handled using masked losses during multi-task training.

In [ ]:
from pathlib import Path
import pandas as pd


# Audio recordings we want to use as the primary CoughSense input.
COUGH_TYPES = ["cough-heavy", "cough-shallow"]


def normalize_smoker(value):
    """
    Normalize the inconsistent smoking-status values.

    Returns
    -------
    1 : known smoker
    0 : known non-smoker
    None : unknown or missing
    """

    if pd.isna(value):
        return None

    if value is True or value == "y":
        return 1

    if value is False or value == "n":
        return 0

    return None


def build_split(dataset_dir, split):
    """
    Build one clean CoughSense dataset split.

    Steps:
    1. Load participant-level metadata.
    2. Find all audio Parquet shards for the split.
    3. Keep cough-heavy and cough-shallow recordings.
    4. Remove recordings with quality score 0.
    5. Match audio recordings to participant metadata.
    6. Normalize smoking status.
    """

    dataset_dir = Path(dataset_dir)

    # Paths for this dataset split.
    audio_dir = dataset_dir / "audio" / split
    metadata_path = dataset_dir / "metadata" / f"{split}.parquet"

    # Metadata fields we currently need.
    metadata_columns = [
        "id",
        "a",                  # age
        "g",                  # gender
        "smoker",
        "asthma",
        "pneumonia",
        "fever",
        "loss_of_smell",
        "covid_status",
        "ihd",
    ]

    # Load participant-level metadata.
    metadata = pd.read_parquet(
        metadata_path,
        columns=metadata_columns,
    )

    # Find every audio Parquet shard in this split.
    audio_files = sorted(audio_dir.glob("*.parquet"))

    if not audio_files:
        raise FileNotFoundError(
            f"No audio Parquet files found in: {audio_dir}"
        )

    audio_frames = []

    # Process every audio shard.
    for audio_file in audio_files:

        # Read only the audio columns we need.
        frame = pd.read_parquet(
            audio_file,
            columns=[
                "participant_id",
                "audio_type",
                "audio",
                "quality_score",
            ],
        )

        # Keep only cough-heavy and cough-shallow recordings.
        frame = frame[
            frame["audio_type"].isin(COUGH_TYPES)
        ].copy()

        # According to the dataset documentation:
        # 0 = bad quality and excluded from classification.
        # 1 = good quality.
        # 2 = excellent quality.
        frame = frame[
            frame["quality_score"] >= 1
        ].copy()

        audio_frames.append(frame)

    # Combine all audio shards into one DataFrame.
    audio = pd.concat(
        audio_frames,
        ignore_index=True,
    )

    # Match each audio recording to its participant metadata.
    #
    # Audio uses participant_id.
    # Metadata uses id.
    result = audio.merge(
        metadata,
        left_on="participant_id",
        right_on="id",
        how="inner",
        validate="many_to_one",
    )

    # Create a clean smoking-status feature.
    result["smoker_clean"] = result["smoker"].apply(
        normalize_smoker
    )

    return result

In [ ]:
train_df = build_split(DATASET_DIR, "train")

print("Training rows:", len(train_df))

print("\nColumns:")
print(train_df.columns.tolist())

display(train_df.head())

## 26. Build Train, Validation, and Test Splits

The same dataset-building procedure is applied independently to the training, validation, and test splits.

The original participant-level split is preserved to prevent data leakage between model development and final evaluation.

In [ ]:
val_df = build_split(DATASET_DIR, "val")
test_df = build_split(DATASET_DIR, "test")

print("Train rows:", len(train_df))
print("Validation rows:", len(val_df))
print("Test rows:", len(test_df))

print("\nUnique participants:")
print("Train:", train_df["participant_id"].nunique())
print("Validation:", val_df["participant_id"].nunique())
print("Test:", test_df["participant_id"].nunique())

## 27. Create CoughSense Prediction Targets

The original metadata uses `True` for positive annotations and `None` when an annotation is unavailable.

For the CoughSense multi-task model, the explicitly positive annotations are converted to `1`, while missing annotations remain missing.

This allows each prediction head to use only participants with a known target instead of incorrectly treating missing annotations as negative.

In [ ]:
TARGET_COLUMNS = [
    "asthma",
    "pneumonia",
    "fever",
    "loss_of_smell",
]

for df in [train_df, val_df, test_df]:
    for target in TARGET_COLUMNS:
        df[f"{target}_label"] = df[target].map(
            {True: 1}
        )

In [ ]:
for target in TARGET_COLUMNS:
    print(f"\n--- {target} ---")
    print(
        train_df[f"{target}_label"]
        .value_counts(dropna=False)
    )

## 28. Create the COVID-Related Screening Target

The original `covid_status` field contains multiple categories.

For the CoughSense screening task, the three explicitly positive categories are grouped into a positive COVID-related label:

- `positive_mild`
- `positive_moderate`
- `positive_asymp`

The remaining resolved non-positive categories are grouped as negative for this screening task:

- `healthy`
- `no_resp_illness_exposed`
- `recovered_full`
- `resp_illness_not_identified`

Records marked `under_validation` remain unlabeled and will be excluded from the COVID-specific loss.

In [ ]:
COVID_POSITIVE = {
    "positive_mild",
    "positive_moderate",
    "positive_asymp",
}

COVID_NEGATIVE = {
    "healthy",
    "no_resp_illness_exposed",
    "recovered_full",
    "resp_illness_not_identified",
}


def create_covid_label(value):
    if value in COVID_POSITIVE:
        return 1

    if value in COVID_NEGATIVE:
        return 0

    # Includes under_validation and any unexpected/missing value.
    return float("nan")


for df in [train_df, val_df, test_df]:
    df["covid_label"] = df["covid_status"].apply(
        create_covid_label
    )

In [ ]:
print(
    train_df["covid_label"]
    .value_counts(dropna=False)
)

## 29. Convert Cough Audio to a Mel Spectrogram

The cough recordings are stored as WAV audio bytes inside the Parquet files.

For the CNN, each recording will be converted from its waveform into a Mel spectrogram. This representation shows how the energy of different frequency components changes over time and provides an image-like representation suitable for convolutional neural networks.

In [ ]:
import io

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf


# Take the first usable cough recording from the training set.
sample_audio = train_df.iloc[0]["audio"]

# Extract the WAV bytes stored inside the dictionary.
audio_bytes = sample_audio["bytes"]

# Read the WAV data directly from memory.
waveform, sample_rate = sf.read(io.BytesIO(audio_bytes))

print("Sample rate:", sample_rate)
print("Waveform shape:", waveform.shape)
print("Duration (seconds):", len(waveform) / sample_rate)

In [ ]:
# Generate a Mel spectrogram from the decoded waveform.
mel_spectrogram = librosa.feature.melspectrogram(
    y=waveform.astype(np.float32),
    sr=sample_rate,
    n_mels=128,
    fmax=sample_rate // 2,
)

# Convert the power spectrogram to decibels.
mel_db = librosa.power_to_db(
    mel_spectrogram,
    ref=np.max,
)

print("Mel spectrogram shape:", mel_db.shape)

### 29.1 Visualize a Cough Mel Spectrogram

We visualize the generated Mel spectrogram to verify that the audio preprocessing pipeline produces a meaningful time-frequency representation before applying it to the complete dataset.

In [ ]:
plt.figure(figsize=(10, 4))

librosa.display.specshow(
    mel_db,
    sr=sample_rate,
    x_axis="time",
    y_axis="mel",
)

plt.colorbar(format="%+2.0f dB")
plt.title("Cough Mel Spectrogram")
plt.tight_layout()
plt.show()

## 30. Create the Audio Preprocessing Function

Each cough recording is converted from WAV bytes into a Mel spectrogram.

The preprocessing function will be reused during training and inference so that the model receives the same type of input in both situations.

Spectrograms will be generated in memory rather than stored as thousands of image files.

In [ ]:
import io

import librosa
import numpy as np
import soundfile as sf

def audio_to_melspectrogram(audio_dict, target_frames=256):
    """
    Convert WAV audio bytes into a fixed-size Mel spectrogram.

    Short spectrograms are padded along the time axis.
    Long spectrograms are cropped.
    """

    # Extract WAV bytes from the Parquet audio dictionary.
    audio_bytes = audio_dict["bytes"]

    # Decode the WAV audio.
    waveform, sample_rate = sf.read(
        io.BytesIO(audio_bytes)
    )

    # Convert stereo recordings to mono.
    if waveform.ndim > 1:
        waveform = waveform.mean(axis=1)

    # Generate the Mel spectrogram.
    mel = librosa.feature.melspectrogram(
        y=waveform.astype(np.float32),
        sr=sample_rate,
        n_mels=128,
        fmax=sample_rate // 2,
    )

    # Convert power values to decibels.
    mel_db = librosa.power_to_db(
        mel,
        ref=np.max,
    ).astype(np.float32)

    # Current number of time frames.
    current_frames = mel_db.shape[1]

    # Pad shorter recordings.
    if current_frames < target_frames:
        pad_width = target_frames - current_frames

        mel_db = np.pad(
            mel_db,
            ((0, 0), (0, pad_width)),
            mode="constant",
            constant_values=mel_db.min(),
        )

    # Crop longer recordings.
    elif current_frames > target_frames:
        mel_db = mel_db[:, :target_frames]

    return mel_db

In [ ]:
test_mel = audio_to_melspectrogram(
    train_df.iloc[0]["audio"]
)

print("Spectrogram shape:", test_mel.shape)
print("Data type:", test_mel.dtype)
print("Minimum:", test_mel.min())
print("Maximum:", test_mel.max())

## 31. Create the PyTorch Dataset

The PyTorch Dataset connects the cleaned CoughSense DataFrame to the machine learning model.

For each recording, it will provide:

- a Mel spectrogram for the CNN
- contextual features such as age, gender, and smoking status for the MLP
- the available multi-task target labels

Missing target labels will remain masked and will not contribute to the corresponding training loss.

In [ ]:
import torch
from torch.utils.data import Dataset


class CoughSenseDataset(Dataset):
    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]

        # Convert the cough audio into a Mel spectrogram.
        mel = audio_to_melspectrogram(row["audio"])

        # Add the channel dimension expected by a CNN.
        mel = torch.tensor(mel, dtype=torch.float32).unsqueeze(0)

        # Contextual features for the MLP.
        age = float(row["a"])

        gender = row["g"]

        if gender == "male":
            gender_encoded = [1.0, 0.0, 0.0]
        elif gender == "female":
            gender_encoded = [0.0, 1.0, 0.0]
        else:
            gender_encoded = [0.0, 0.0, 1.0]

        smoker = row["smoker_clean"]

        if pd.isna(smoker):
            smoker_value = 0.0
            smoker_missing = 1.0
        else:
            smoker_value = float(smoker)
            smoker_missing = 0.0

        metadata_features = torch.tensor(
            [age / 100.0, *gender_encoded, smoker_value, smoker_missing],
            dtype=torch.float32,
        )

        
        # Primary CoughSense screening target.
        screening_label = row["respiratory_concern"]

        if pd.isna(screening_label):
            screening_label = -1.0
        else:
            screening_label = float(screening_label)

        labels = torch.tensor(
            [screening_label],
            dtype=torch.float32,
)

        return {
            "spectrogram": mel,
            "metadata": metadata_features,
            "labels": labels,
        }

In [ ]:
train_dataset = CoughSenseDataset(train_df)

sample = train_dataset[0]

print("Spectrogram:", sample["spectrogram"].shape)
print("Metadata:", sample["metadata"].shape)
print("Labels:", sample["labels"])

## 32. CoughSense CNN + MLP Model

The CoughSense model uses two branches:

- A CNN processes the Mel spectrogram and learns acoustic representations from the cough.
- An MLP processes contextual features such as age, gender, and smoking status.

The two representations are combined and passed to separate prediction heads for each screening target.

Each target is learned independently so that missing annotations can be ignored during training.

In [ ]:
import torch
import torch.nn as nn


class CoughSenseModel(nn.Module):
    def __init__(self, metadata_dim=6, num_targets=1):
        super().__init__()

        # -------------------------
        # CNN branch: audio
        # -------------------------
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.audio_projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        # -------------------------
        # MLP branch: metadata
        # -------------------------
        self.metadata_mlp = nn.Sequential(
            nn.Linear(metadata_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 32),
            nn.ReLU(),
        )

        # -------------------------
        # Fusion
        # -------------------------
        self.fusion = nn.Sequential(
            nn.Linear(128 + 32, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
        )

        # -------------------------
        # Multi-task prediction heads
        # -------------------------
        self.heads = nn.ModuleList([
            nn.Linear(64, 1) for _ in range(num_targets)
        ])

    def forward(self, spectrogram, metadata):
        audio_features = self.cnn(spectrogram)
        audio_features = self.audio_projection(audio_features)

        metadata_features = self.metadata_mlp(metadata)

        combined = torch.cat(
            [audio_features, metadata_features],
            dim=1,
        )

        fused = self.fusion(combined)

        outputs = torch.cat(
            [head(fused) for head in self.heads],
            dim=1,
        )

        return outputs

In [ ]:
model = CoughSenseModel()

sample_spectrogram = sample["spectrogram"].unsqueeze(0)
sample_metadata = sample["metadata"].unsqueeze(0)

output = model(
    sample_spectrogram,
    sample_metadata,
)

print("Output shape:", output.shape)
print("Output:", output)

## 33. Masked Multi-Task Loss

The CoughSense dataset contains incomplete annotations. A value of `-1` represents an unknown target and should not contribute to the loss.

For each prediction head, binary cross-entropy loss is calculated only for samples with a known target.

This allows the model to learn different tasks from different subsets of the annotated training data without incorrectly treating missing labels as negative examples.

In [ ]:
import torch
import torch.nn.functional as F


def masked_multitask_loss(outputs, targets):
    """
    Calculate binary cross-entropy loss while ignoring
    unknown targets represented by -1.
    """

    total_loss = 0.0
    valid_tasks = 0

    for task_index in range(outputs.shape[1]):
        task_outputs = outputs[:, task_index]
        task_targets = targets[:, task_index]

        # Only use samples with known labels.
        mask = task_targets != -1

        if mask.any():
            loss = F.binary_cross_entropy_with_logits(
                task_outputs[mask],
                task_targets[mask],
            )

            total_loss += loss
            valid_tasks += 1

    if valid_tasks == 0:
        return outputs.sum() * 0.0

    return total_loss / valid_tasks

In [ ]:
loss = masked_multitask_loss(
    output,
    sample["labels"].unsqueeze(0),
)

print("Loss:", loss.item())

## 34. Create DataLoaders

PyTorch DataLoaders group individual CoughSense samples into batches for efficient model training.

The training DataLoader will shuffle the training samples, while validation and test DataLoaders will preserve their ordering.

In [ ]:
from torch.utils.data import DataLoader

train_dataset = CoughSenseDataset(train_df)
val_dataset = CoughSenseDataset(val_df)
test_dataset = CoughSenseDataset(test_df)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

In [ ]:
batch = next(iter(train_loader))

print("Spectrogram:", batch["spectrogram"].shape)
print("Metadata:", batch["metadata"].shape)
print("Labels:", batch["labels"].shape)

## 35. Train the CoughSense Model

The CoughSense model is trained using the prepared training DataLoader.

For each batch, the CNN processes the Mel spectrogram, the MLP processes contextual features, and the resulting representations are fused before producing five task-specific outputs.

The masked multi-task loss ensures that missing target annotations do not contribute to the corresponding task's loss.

## 35. Train the CoughSense Model

The model is trained batch-by-batch using the training DataLoader. The CNN processes cough spectrograms, the MLP processes contextual features, and the masked multi-task loss allows each prediction head to learn only from samples with an available label.

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()

    total_loss = 0.0

    for batch in loader:
        spectrograms = batch["spectrogram"].to(device)
        metadata = batch["metadata"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            spectrograms,
            metadata
        )

        loss = masked_multitask_loss(
            outputs,
            labels
        )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
def validate(model, loader, device):
    model.eval()

    total_loss = 0.0

    with torch.no_grad():
        for batch in loader:
            spectrograms = batch["spectrogram"].to(device)
            metadata = batch["metadata"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                spectrograms,
                metadata
            )

            loss = masked_multitask_loss(
                outputs,
                labels
            )

            total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

model = CoughSenseModel(
    metadata_dim=6,
    num_targets=5,
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
)

In [ ]:
train_loss = train_one_epoch(
    model,
    train_loader,
    optimizer,
    device,
)

val_loss = validate(
    model,
    val_loader,
    device,
)

print("Train loss:", train_loss)
print("Validation loss:", val_loss)

In [ ]:
print("Missing audio:", train_df["audio"].isna().sum())
print("Total training rows:", len(train_df))

### 35.1 Remove Missing Audio Samples

A small number of records contain missing audio data and therefore cannot be converted into Mel spectrograms.

These records are removed from the corresponding dataset because an audio sample is required for CoughSense's CNN input. Missing medical labels are handled separately using masked losses and are not treated as reasons to remove the sample.

In [ ]:
print("Training rows before:", len(train_df))

train_df = train_df.dropna(subset=["audio"]).reset_index(drop=True)

print("Training rows after:", len(train_df))
print("Removed:", 3579 - len(train_df))

In [ ]:
print("Validation missing audio:", val_df["audio"].isna().sum())
print("Test missing audio:", test_df["audio"].isna().sum())

### 35.2 Remove Missing Audio from Validation

Validation records without audio cannot be passed through the CNN and are therefore removed before evaluation.

The test set contains no missing audio records and does not require this cleanup.

In [ ]:
print("Validation rows before:", len(val_df))

val_df = val_df.dropna(subset=["audio"]).reset_index(drop=True)

print("Validation rows after:", len(val_df))
print("Removed:", 1)

## Recreate Dataset and Dataloaders


In [ ]:
train_dataset = CoughSenseDataset(train_df)
val_dataset = CoughSenseDataset(val_df)
test_dataset = CoughSenseDataset(test_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

In [ ]:
batch = next(iter(train_loader))

print("Spectrogram:", batch["spectrogram"].shape)
print("Metadata:", batch["metadata"].shape)
print("Labels:", batch["labels"].shape)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

model = CoughSenseModel(
    metadata_dim=6,
    num_targets=5
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [ ]:
train_loss = train_one_epoch(
    model,
    train_loader,
    optimizer,
    device
)

val_loss = validate(
    model,
    val_loader,
    device
)

print("Train loss:", train_loss)
print("Validation loss:", val_loss)

## 36. Train the Final CoughSense Model

After verifying that the complete training pipeline works for one epoch, the model is trained for multiple epochs.

Validation loss is monitored after each epoch so that the best-performing model can be saved for later evaluation and deployment.

In [ ]:
best_val_loss = float("inf")
best_model_state = None

num_epochs = 10

for epoch in range(num_epochs):
    train_loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        device
    )

    val_loss = validate(
        model,
        val_loader,
        device
    )

    print(
        f"Epoch {epoch + 1}/{num_epochs} "
        f"| Train Loss: {train_loss:.4f} "
        f"| Val Loss: {val_loss:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }

## Restore the Best Model

In [ ]:
model.load_state_dict(best_model_state)
model.to(device)
model.eval()

In [ ]:
model.load_state_dict(best_model_state)
model.to(device)
model.eval()

print("Best model restored.")
print("Best validation loss:", best_val_loss)

## 37. Final Test Evaluation

The model with the lowest validation loss is evaluated on the held-out test set.

The test set was kept separate throughout training and model selection. Performance is calculated independently for each CoughSense prediction target.

In [ ]:
import numpy as np
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

TARGET_NAMES = [
    "Asthma",
    "Pneumonia",
    "Fever",
    "Loss of smell",
    "COVID-related",
]


def collect_test_predictions(model, loader, device):
    model.eval()

    all_outputs = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            spectrograms = batch["spectrogram"].to(device)
            metadata = batch["metadata"].to(device)

            outputs = model(
                spectrograms,
                metadata,
            )

            all_outputs.append(outputs.cpu())
            all_labels.append(batch["labels"])

    outputs = torch.cat(all_outputs).numpy()
    labels = torch.cat(all_labels).numpy()

    return outputs, labels


test_outputs, test_labels = collect_test_predictions(
    model,
    test_loader,
    device,
)

print("Test output shape:", test_outputs.shape)
print("Test label shape:", test_labels.shape)

In [ ]:
test_probabilities = 1 / (1 + np.exp(-test_outputs))

for i, target_name in enumerate(TARGET_NAMES):
    y_true = test_labels[:, i]
    y_prob = test_probabilities[:, i]

    # Ignore unknown labels.
    mask = y_true != -1

    y_true = y_true[mask].astype(int)
    y_prob = y_prob[mask]

    if len(np.unique(y_true)) < 2:
        print(f"\n--- {target_name} ---")
        print("Not enough classes for evaluation.")
        continue

    y_pred = (y_prob >= 0.5).astype(int)

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0,
    )

    auc = roc_auc_score(
        y_true,
        y_prob,
    )

    print(f"\n--- {target_name} ---")
    print("Known test samples:", len(y_true))
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1:        {f1:.4f}")
    print(f"ROC-AUC:   {auc:.4f}")

## 38. Create the Respiratory Screening Target

The individual symptom and condition annotations are sparse and often missing, so they are not treated as independent negative labels.

For the primary CoughSense screening task, the available annotations are combined into a single respiratory-concern target.

A sample is labelled positive when a recorded respiratory concern is present, including a positive COVID-related status or a positive annotation for asthma, pneumonia, fever, or loss of smell.

A sample is labelled as having no recorded concern when the participant is marked as healthy and none of the selected respiratory indicators are positive.

All remaining ambiguous or insufficiently annotated records remain unlabeled and are excluded from the screening loss.

This target is intended for preliminary screening rather than medical diagnosis.

In [ ]:
SCREENING_POSITIVE_COVID = {
    "positive_mild",
    "positive_moderate",
    "positive_asymp",
}

SCREENING_NEGATIVE_COVID = {
    "healthy",
}


def create_respiratory_concern_label(row):
    """
    Create the primary CoughSense screening label.

    Returns:
        1 -> recorded respiratory concern
        0 -> no recorded concern
        NaN -> insufficient or ambiguous information
    """

    # Any explicitly recorded respiratory indicator is positive.
    symptom_columns = [
        "asthma",
        "pneumonia",
        "fever",
        "loss_of_smell",
    ]

    for column in symptom_columns:
        if row[column] is True:
            return 1

    # Positive COVID-related status is a respiratory concern.
    if row["covid_status"] in SCREENING_POSITIVE_COVID:
        return 1

    # A healthy COVID status with no positive indicator gives us
    # a "no recorded concern" example.
    if row["covid_status"] in SCREENING_NEGATIVE_COVID:
        return 0

    # Everything else is ambiguous/insufficiently labelled.
    return float("nan")


for df in [train_df, val_df, test_df]:
    df["respiratory_concern"] = df.apply(
        create_respiratory_concern_label,
        axis=1,
    )

In [ ]:
for name, df in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df),
]:
    print(f"\n--- {name} ---")
    print(
        df["respiratory_concern"]
        .value_counts(dropna=False)
        .sort_index()
        .to_string()
    )

## 39. Rebuild DataLoaders for Respiratory Screening

In [ ]:
train_dataset = CoughSenseDataset(train_df)
val_dataset = CoughSenseDataset(val_df)
test_dataset = CoughSenseDataset(test_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
)

In [ ]:
batch = next(iter(train_loader))

print("Spectrogram:", batch["spectrogram"].shape)
print("Metadata:", batch["metadata"].shape)
print("Labels:", batch["labels"].shape)

## 40. Initialize the Respiratory Screening Model

The primary CoughSense model predicts a single respiratory-concern score.

The model combines acoustic features extracted from the cough Mel spectrogram with contextual participant features before producing the final screening output.

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)

screening_model = CoughSenseModel(
    metadata_dim=6,
    num_targets=1,
).to(device)

screening_optimizer = torch.optim.Adam(
    screening_model.parameters(),
    lr=1e-3,
)

## 41. Train the Respiratory Screening Model

The screening model is first trained for one epoch to verify that the new aggregated respiratory-concern target works correctly with the complete training pipeline.

In [ ]:
screening_train_loss = train_one_epoch(
    screening_model,
    train_loader,
    screening_optimizer,
    device,
)

screening_val_loss = validate(
    screening_model,
    val_loader,
    device,
)

print("Train loss:", screening_train_loss)
print("Validation loss:", screening_val_loss)

In [ ]:
best_screening_val_loss = float("inf")
best_screening_model_state = None

num_epochs = 10

for epoch in range(num_epochs):

    train_loss = train_one_epoch(
        screening_model,
        train_loader,
        screening_optimizer,
        device,
    )

    val_loss = validate(
        screening_model,
        val_loader,
        device,
    )

    print(
        f"Epoch {epoch + 1}/{num_epochs} "
        f"| Train Loss: {train_loss:.4f} "
        f"| Val Loss: {val_loss:.4f}"
    )

    if val_loss < best_screening_val_loss:
        best_screening_val_loss = val_loss

        best_screening_model_state = {
            key: value.detach().cpu().clone()
            for key, value in screening_model.state_dict().items()
        }

In [ ]:
screening_model.load_state_dict(
    best_screening_model_state
)

screening_model.to(device)
screening_model.eval()

print("Best screening model restored.")
print("Best validation loss:", best_screening_val_loss)

## 44. Final Test Evaluation

The best respiratory screening model is evaluated on the held-out test set.

Unknown labels are excluded from metric calculations. The final evaluation reports precision, recall, F1-score, and ROC-AUC.

In [ ]:
import numpy as np
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

screening_model.eval()

all_outputs = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        spectrograms = batch["spectrogram"].to(device)
        metadata = batch["metadata"].to(device)

        outputs = screening_model(
            spectrograms,
            metadata,
        )

        all_outputs.append(outputs.cpu())
        all_labels.append(batch["labels"])

test_outputs = torch.cat(all_outputs).numpy().flatten()
test_labels = torch.cat(all_labels).numpy().flatten()

# Ignore unknown labels.
mask = test_labels != -1

y_true = test_labels[mask].astype(int)
y_prob = 1 / (1 + np.exp(-test_outputs[mask]))

# Convert probabilities to binary predictions.
y_pred = (y_prob >= 0.5).astype(int)

print("Known test samples:", len(y_true))
print("Positive samples:", (y_true == 1).sum())
print("Negative samples:", (y_true == 0).sum())

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0,
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0,
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0,
)

auc = roc_auc_score(
    y_true,
    y_prob,
)

print("\n--- CoughSense Respiratory Screening ---")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")
print(f"ROC-AUC:   {auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

In [ ]:
import os

os.makedirs("models", exist_ok=True)

screening_model.load_state_dict(
    best_screening_model_state
)

torch.save(
    screening_model.state_dict(),
    "models/coughsense_screening_model.pth"
)

print("Model saved successfully.")

In [ ]:
import os

print(
    "Model exists:",
    os.path.exists("models/coughsense_screening_model.pth")
)

print(
    "Model size:",
    os.path.getsize(
        "models/coughsense_screening_model.pth"
    ) / (1024 * 1024),
    "MB"
)

In [ ]:
import os

os.makedirs("models", exist_ok=True)

torch.save(
    screening_model.state_dict(),
    "models/coughsense_screening_model.pth"
)

print(
    os.path.exists(
        "models/coughsense_screening_model.pth"
    )
)

In [ ]:
import os
import torch

os.makedirs("../models", exist_ok=True)

torch.save(
    screening_model.state_dict(),
    "../models/coughsense_screening_model.pth"
)

print("Saved:", os.path.exists(
    "../models/coughsense_screening_model.pth"
))